# Using the Titanic RandomForest Baseline model

Loads the `rf-small` variation of [Titanic RandomForest Baseline](https://www.kaggle.com/models/tranphunghia233871/titanic-randomforest-baseline) and predicts survival. The model was fitted on `train.csv`, so the accuracy below is training accuracy, not an estimate of leaderboard performance.

In [ ]:
import glob
import pickle

import pandas as pd

model_path = glob.glob("/kaggle/input/**/model.pkl", recursive=True)[0]
train_path = glob.glob("/kaggle/input/**/train.csv", recursive=True)[0]
with open(model_path, "rb") as f:
    model = pickle.load(f)
print(model_path, train_path)
model

## Prepare the inputs
The model expects the same seven encoded features it was trained on.

In [ ]:
FEATURES = ["Pclass", "Sex", "Age", "SibSp", "Parch", "Fare", "Embarked"]


def prepare(df):
    x = df[FEATURES].copy()
    x["Sex"] = (x["Sex"] == "female").astype(int)
    x["Embarked"] = x["Embarked"].fillna("S").map({"S": 0, "C": 1, "Q": 2})
    return x.fillna({"Age": 28.0, "Fare": 14.45})


train = pd.read_csv(train_path)
X = prepare(train)
X.head()

## Predict

In [ ]:
print("training accuracy:", round(model.score(X, train["Survived"]), 4))
train.assign(survival_probability=model.predict_proba(X)[:, 1])[["Name", "Sex", "Pclass", "survival_probability"]].head()

## Which features matter?

In [ ]:
pd.Series(model.feature_importances_, index=FEATURES).sort_values(ascending=False).round(3)